# The Agent Loop

A streaming LLM client ([NB01](/notebooks/apps/cda/01-client.html)) and a tool system ([NB02](/notebooks/apps/cda/02-tools.html)) are necessary but not sufficient for a coding agent. The client can talk to a model and the registry can execute tools, but nothing connects them into the multi-turn cycle that makes an agent *agentic*: the model thinks, decides to call a tool, the runtime executes it, feeds the result back, and the model thinks again — repeating until the task is done.

This notebook builds that connection in three layers: (i) **system prompts** that tell the model its identity, environment, available tools, and operational guidelines; (ii) a **Session** that wires the client and registry to a shared message history with token tracking; and (iii) the **Agent** class whose `run()` method implements the agentic loop as an async generator of typed events. The result: `async for event in agent.run("Fix the bug"): ...` — a single call that orchestrates an entire multi-turn coding session.

## System Prompts

The system prompt is the agent's constitution — it defines identity, capabilities, constraints, and behavioral norms. A poorly structured prompt produces an agent that ignores tools, leaks secrets, or rambles. We assemble the prompt from **modular sections** so that each concern can be maintained independently. The `build_system_prompt(config, tools)` function takes the current config and the list of registered tools and returns the complete prompt string.

In [ ]:
from notebooks.agent.config import Config
from notebooks.agent.tools.registry import create_default_registry
from notebooks.agent.prompts import build_system_prompt

config = Config()
registry = create_default_registry(config)
tools = registry.get_tools()

prompt = build_system_prompt(config, tools)
print(f"System prompt: {len(prompt):,} characters, {len(prompt.split())} words")

The prompt is assembled from independent sections, each handling one concern:

In [ ]:
for line in prompt.splitlines():
    if line.startswith("# "):
        print(f"  {line}")

- **Identity** — "You are an AI coding agent... pair-programming with the user."
- **Environment** — date, OS, shell, working directory (from `config` + `platform`).
- **Available Tools** — formatted list of registered tools plus best practices (read before editing, search before acting, surgical edits, parallelism).
- **Security Guidelines** — never expose secrets, validate paths, cautious commands, prompt-injection defense.
- **Operational Guidelines** — tone (concise, direct), workflow (understand → plan → implement → verify), error recovery.

**Identity and Environment.** The first two sections anchor the model's behavior and provide runtime context:

In [ ]:
sections = prompt.split("\n\n# ")
identity = sections[0]
environment = "# " + sections[1]

print(identity)
print()
print(environment)

**Available Tools.** The model gets a formatted list of every registered tool with its description, plus usage guidelines:

In [ ]:
tools_section = "# " + sections[2]
for line in tools_section.splitlines()[:20]:
    print(line)

:::{.callout-note}
The `developer_instructions` field is for project-level context (e.g., "This project uses Ruff for linting"), while `user_instructions` is for per-user preferences. Both are optional — when set, they get their own sections in the system prompt. This mirrors the approach used by Claude Code and similar production agents.

:::

In [ ]:
custom_config = Config(
    developer_instructions="This project uses Ruff for linting. Always run `ruff check .` after edits.",
    user_instructions="Prefer concise explanations. Use type hints everywhere.",
)
custom_prompt = build_system_prompt(custom_config, tools)

for line in custom_prompt.splitlines():
    if line.startswith("# "):
        print(f"  {line}")

## Session Management

The `Session` is a **state container** for a single agent conversation. It is *not* the brain — it does not decide what to do next. Its responsibilities: (i) **wiring** the `LLMClient`, `ToolRegistry`, and message list so the agent doesn't manage plumbing; (ii) **message management** via `add_user_message`, `add_assistant_message`, and `add_tool_result` that maintain the conversation in the exact format the OpenAI API expects; (iii) **token tracking** across all LLM calls in the session; and (iv) **initialization** — building the system prompt on construction.

In [ ]:
from notebooks.agent.session import Session

session = Session(config)

print(f"Messages:   {len(session.messages)} (system prompt)")
print(f"Turn count: {session.turn_count}")
print(f"Usage:      {session.total_usage.total_tokens} tokens")
print(f"Tools:      {len(session.get_tool_schemas())} schemas ready")
print(f"Prompt:     {len(session.system_prompt):,} chars")

**Message lifecycle.** We simulate a conversation turn to show how the four message types build up:

In [ ]:
from notebooks.agent.events import TokenUsage, ToolResultMessage

session.add_user_message("Read main.py and add type hints")

session.add_assistant_message(
    content="I'll read the file first.",
    tool_calls=[{
        "id": "call_001",
        "name": "read_file",
        "arguments": {"path": "src/main.py"},
    }],
)

session.add_tool_result(ToolResultMessage(
    tool_call_id="call_001",
    content="1|def main():\n2|    print('hello')\n",
))

session.track_usage(TokenUsage(prompt_tokens=500, completion_tokens=50, total_tokens=550))

print(f"Messages: {len(session.messages)}")
for msg in session.messages:
    role = msg["role"]
    content = str(msg.get("content", ""))[:60]
    tc = "  + tool_calls" if "tool_calls" in msg else ""
    print(f"  [{role:10s}] {content}...{tc}")

The four message types in the conversation: (1) `system` — the assembled prompt (always first), (2) `user` — the human's request, (3) `assistant` — model's reply, optionally with a `tool_calls` array in OpenAI format, (4) `tool` — result of executing a tool, linked back via `tool_call_id`. The `add_assistant_message` method handles the tedious JSON formatting so the agent loop doesn't have to.

In [ ]:
import json

# The assistant message with tool_calls — the most complex one
assistant_msg = session.messages[2]
print(json.dumps(assistant_msg, indent=2))

In [ ]:
session.reset()
print(f"After reset: {len(session.messages)} messages, turn_count={session.turn_count}")
print(f"System prompt preserved: {session.messages[0]['role'] == 'system'}")

## The Agent

The `Agent` class has a single public method — `run(user_message)` — which is an async generator yielding `AgentEvent` objects. The caller never calls the LLM or executes tools directly; it just iterates over events and renders them. Two layers:

- **`run()`** (outer) — bookkeeping: emits `AGENT_START`, delegates to the loop, emits `AGENT_END` or `AGENT_ERROR`. Catches all exceptions so the event stream is always clean.
- **`_agentic_loop()`** (inner) — the multi-turn cycle:

```
User message
    │
    ▼
┌──────────────────────────────┐
│  LLM call (stream response)  │◄─────────┐
└──────────────────────────────┘           │
    │                                      │
    ├── text only? ──► DONE (final answer) │
    │                                      │
    └── tool calls? ──► Execute tools ─────┘
                        (results added to
                         message history)
```

Each iteration: (1) call the LLM with current messages + tool schemas, (2) stream the response, accumulating text and tool calls, (3) if no tool calls → return (final answer), (4) if tool calls → execute each one, append results, loop back. Bounded by `config.max_turns` to prevent runaway agents.

**AgentEventType.** The seven event types the loop yields — the contract between the agent and any consumer:

| Event | When | `data` keys |
|---|---|---|
| `AGENT_START` | `run()` begins | `message` |
| `TEXT_DELTA` | Each text chunk from LLM | `content` |
| `TEXT_COMPLETE` | Full text accumulated | `content` |
| `TOOL_CALL_START` | Before executing a tool | `call_id`, `name`, `arguments` |
| `TOOL_CALL_COMPLETE` | After tool returns | `call_id`, `name`, `success`, `output`, `error`, `diff`, ... |
| `AGENT_END` | Task complete | `response`, `usage` |
| `AGENT_ERROR` | Something went wrong | `error` |

In [ ]:
from notebooks.agent.events import AgentEvent, AgentEventType

for evt in AgentEventType:
    print(f"  {evt.value}")

start = AgentEvent.agent_start("Fix the bug")
delta = AgentEvent.text_delta("Hello")
print(f"\nExample: type={start.type.value}, data={start.data}")
print(f"Example: type={delta.type.value}, data={delta.data}")

In [ ]:
from notebooks.agent.agent import Agent

agent = Agent(config)

print(f"Agent ready")
print(f"  model:     {agent.config.model_name}")
print(f"  max_turns: {agent.config.max_turns}")
print(f"  cwd:       {agent.config.cwd}")
print(f"  tools:     {len(agent.session.get_tool_schemas())}")

### Simple test run

Before the big demo, we verify the event stream with a simple non-tool-calling query. This confirms the `AGENT_START → TEXT_DELTA* → TEXT_COMPLETE → AGENT_END` lifecycle:

We stream a short geography question and print each event kind as it arrives:

In [ ]:
async for event in agent.run("What is the capital of France? Answer in one sentence."):
    match event.type:
        case AgentEventType.AGENT_START:
            print(f"[START] {event.data['message']}")
        case AgentEventType.TEXT_DELTA:
            print(event.data["content"], end="", flush=True)
        case AgentEventType.TEXT_COMPLETE:
            print()
        case AgentEventType.AGENT_END:
            usage = event.data.get("usage", {})
            print(f"[END] tokens={usage.get('total_tokens', 'N/A')}")
        case AgentEventType.AGENT_ERROR:
            print(f"[ERROR] {event.data['error']}")

## Live Demo: End-to-End Agent Run

Now the real test: we give the agent a coding task and watch the full think → act → observe cycle. The agent should: (1) write a Python file using `write_file`, (2) run it with `shell` to verify, and (3) report back. We create a temporary workspace so it has a clean directory to work in.

In [ ]:
import tempfile
from pathlib import Path
from notebooks.agent.config import ApprovalPolicy

workspace = Path(tempfile.mkdtemp(prefix="agent_demo_"))
demo_config = Config(cwd=workspace, approval=ApprovalPolicy.AUTO)  # <1>
demo_agent = Agent(demo_config)                                     # <2>

print(f"Workspace: {workspace}")

1. `ApprovalPolicy.AUTO` is stored on `Config` for NB04's `ApprovedAgent` to read. The base `Agent` here always executes tools unconditionally — approval wiring is added in [NB04](/notebooks/apps/cda/04-hardening.html).
2. Fresh agent with a session pointing at the temp workspace.

In [ ]:
task = (
    "Create a Python script called fib.py that prints the first 10 Fibonacci numbers, "
    "one per line. Then run it to verify the output is correct."
)

async for event in demo_agent.run(task):
    match event.type:
        case AgentEventType.AGENT_START:
            print(f"{'─' * 60}")
            print(f"  Task: {event.data['message'][:80]}")
            print(f"{'─' * 60}\n")

        case AgentEventType.TEXT_DELTA:
            print(event.data["content"], end="", flush=True)

        case AgentEventType.TEXT_COMPLETE:
            print("\n")

        case AgentEventType.TOOL_CALL_START:
            name = event.data["name"]
            args = event.data["arguments"]
            args_short = str(args)[:80] + ("..." if len(str(args)) > 80 else "")
            print(f"  > {name}({args_short})")

        case AgentEventType.TOOL_CALL_COMPLETE:
            name = event.data["name"]
            ok = "OK" if event.data.get("success") else "FAIL"
            output = (event.data.get("output") or "")[:120]
            print(f"  [{ok}] {name} -> {output}")
            if event.data.get("diff"):
                for line in event.data["diff"].splitlines()[:10]:
                    print(f"    {line}")
            print()

        case AgentEventType.AGENT_END:
            usage = event.data.get("usage", {})
            print(f"{'─' * 60}")
            total = usage.get('total_tokens', 'N/A') if usage else 'N/A'
            print(f"  Done. Tokens: {total}")
            print(f"{'─' * 60}")

        case AgentEventType.AGENT_ERROR:
            print(f"\n  ERROR: {event.data['error']}")

The expected multi-turn execution:

- **Turn 1:** The model receives the user message + system prompt + tool schemas. It calls `write_file` to create `fib.py`. The tool result is added to messages.
- **Turn 2:** The model sees "File written", calls `shell` with `python fib.py` to verify. The output is appended.
- **Turn 3:** The model sees the Fibonacci output, confirms correctness, and produces a final text response (no tool calls). The loop exits.

This is the **think → act → observe** cycle. Each turn, the model sees the *entire* conversation so far, including all tool results.

In [ ]:
fib_path = workspace / "fib.py"
print(f"File exists: {fib_path.exists()}")
if fib_path.exists():
    print(f"\n{fib_path.name}:")
    print(fib_path.read_text())

In [ ]:
s = demo_agent.session
print(f"Total messages: {len(s.messages)}")
print(f"Turn count:     {s.turn_count}")
print(f"Token usage:    {s.total_usage.total_tokens:,} total")
print(f"  prompt:       {s.total_usage.prompt_tokens:,}")
print(f"  completion:   {s.total_usage.completion_tokens:,}")
print(f"  cached:       {s.total_usage.cached_tokens:,}")
print()
print("Message roles:")
for i, msg in enumerate(s.messages):
    role = msg["role"]
    tc = " + tool_calls" if "tool_calls" in msg else ""
    content_preview = str(msg.get("content", ""))[:50].replace("\n", "\\n")
    print(f"  [{i:2d}] {role:10s}{tc:14s}  {content_preview}...")

:::{.callout-note}
The session preserves the full conversation. After `run()` returns, you can call `agent.run("Now add a docstring to the function")` and the model sees all previous messages — including the file it wrote and the shell output from the test run. Each `run()` call extends (not replaces) the conversation history.

:::

### Cleanup

In [ ]:
import shutil

await demo_agent.session.client.close()
shutil.rmtree(workspace)
print(f"Cleaned up {workspace}")

## Summary

Three modules that complete the agent core:

| Module | What it provides |
|--------|------------------|
| `prompts.py` | `build_system_prompt(config, tools)` — modular prompt assembly from sections: identity, environment, tools, security, custom instructions, operational guidelines |
| `session.py` | `Session` — wires `LLMClient` + `ToolRegistry` + message history + `TokenUsage` tracking |
| `agent.py` | `Agent` — `run(user_message)` async generator yielding `AgentEvent` objects; `_agentic_loop()` implements bounded multi-turn think → act → observe cycle |

Key design decisions:

- **Events, not callbacks.** The agent yields a flat stream of `AgentEvent` objects — any consumer (notebook, terminal, Flet app) processes the same stream.
- **Bounded loop.** `config.max_turns` (default 100) prevents runaway agents. The loop emits `AGENT_ERROR` if exhausted.
- **Session is stateful.** Multiple `run()` calls extend the same conversation, enabling multi-turn interaction.
- **Clean separation.** `Session` manages state; `Agent` drives the loop; `prompts.py` constructs the prompt.

The agent works — but it is not yet *safe*. It runs tools without asking, has no context limits, and no loop detection. In the [next notebook](/notebooks/apps/cda/04-hardening.html), we add approval flows, context management, compaction, and loop detection. The [previous notebook](/notebooks/apps/cda/02-tools.html) covers the tool system.